6609612012 ธัญลดา สัมพันธ์ไพศาลสุข

---



# Mini Project - Association Rule Mining

Data source: https://github.com/oilTS/CS372-MiniProject2

GitHub: https://github.com/oilTS/CS372-MiniProject2.git


---



ชุดข้อมูลจำลองพฤติกรรมการซื้อสินค้าอุปโภคบริโภค (Grocery & Staple goods)
ของลูกค้าในร้านค้าปลีกหรือซูเปอร์มาร์เก็ต ประกอบด้วยรายการสินค้าในชีวิตประจำวัน
เช่น น้ำแร่ เนื้อบด สปาเก็ตตี้ วัตถุดิบปรุงอาหาร และขนมขบเคี้ยว


*   ลักษณะและโครงสร้างข้อมูล (Data Structure & Characteristics)

    *   รูปแบบข้อมูล (Transactional Data Format)
        ข้อมูลถูกจัดเก็บในรูปแบบรายการธุรกรรม โดย 1 แถวแทน 1 รายการสั่งซื้อของลูกค้า
    *   ความยาวของข้อมูลไม่คงที่ (Variable Length)
        จำนวนสินค้าในแต่ละ transaction แตกต่างกันตามพฤติกรรมผู้บริโภค
    *   ความกระจัดกระจายของข้อมูล (Data Sparsity)
        ข้อมูลมีลักษณะ sparse เนื่องจากลูกค้าแต่ละรายเลือกซื้อสินค้าเพียงบางส่วนจากสินค้าทั้งหมด

*   ขนาดของข้อมูล (Dataset Size)
ชุดข้อมูลมีประมาณ 7,500 transactions และมีสินค้าให้เลือกมากกว่า 100 รายการ

*   การแปลงข้อมูล (Data Transformation):
หลังจากทำ One-hot Encoding ข้อมูลจะอยู่ในรูปแบบ Binary Matrix (0/1)
ซึ่งเหมาะสำหรับการใช้ algorithm เช่น Apriori และ FP-Growth

*   การกระจายความถี่ของสินค้า (Item Frequency):
มีทั้งสินค้ายอดนิยมและสินค้าที่พบไม่บ่อย
ทำให้สามารถค้นพบทั้ง strong rules และ spurious rules ได้

*   ความเหมาะสมต่อการนำมาวิเคราะห์:
ชุดข้อมูลมีทั้งสินค้ายอดนิยม (Dominant Items)
และสินค้าที่มีการซื้อร่วมกันแบบเฉพาะเจาะจง
จึงเหมาะสำหรับการทำ Association Rule Mining
เพื่อค้นหาความสัมพันธ์เชิงพฤติกรรมของผู้บริโภค
นอกจากนี้ ผลลัพธ์สามารถนำไปใช้ในเชิงธุรกิจได้ เช่น
การจัดโปรโมชั่นสินค้า (Product Bundling),
การจัดวางสินค้าในร้าน (Store Layout)
และระบบแนะนำสินค้า (Recommendation System)

In [6]:
import warnings
warnings.filterwarnings('ignore')

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [12]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# ==============================================
# 1) LOAD DATA FROM GITHUB
# ==============================================
url = "https://raw.githubusercontent.com/oilTS/CS372-MiniProject2/main/Market_Basket_Optimisation.csv"
df = pd.read_csv(url, header=None)

print("========== RAW DATA ==========")
print(df.head())

# ==============================================
# 2) TRANSFORM DATA → TRANSACTION FORMAT
# ==============================================
transactions = []

for i in range(len(df)):
    row = df.iloc[i].dropna().tolist()
    transactions.append(row)

print("\n========== EXAMPLE TRANSACTION ==========")
print(transactions[0])

# ==============================================
# 3) ONE-HOT ENCODING
# ==============================================
te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)
df_encoded = pd.DataFrame(te_array, columns=te.columns_)

print("\n========== ENCODED DATA ==========")
print(df_encoded.head())

# ==============================================
# TEST DIFFERENT SUPPORT
# ==============================================
print("\n========== TEST DIFFERENT SUPPORT ==========")

test_supports = [0.005, 0.01, 0.02]

for s in test_supports:
    temp_items = apriori(df_encoded, min_support=s, use_colnames=True)
    temp_rules = association_rules(temp_items, metric="confidence", min_threshold=0.3)
    print(f"Support = {s} -> #Rules = {len(temp_rules)}")

# ==============================================
# 4) FREQUENT ITEMSETS
# ==============================================
min_sup = 0.01
print("\nChosen min_support =", min_sup)

freq_items = apriori(df_encoded, min_support=min_sup, use_colnames=True)

print("\n========== FREQUENT ITEMSETS ==========")
print(freq_items.head())

# ==============================================
# 5) ASSOCIATION RULES
# ==============================================
min_conf = 0.3
print("Chosen min_confidence =", min_conf)

rules = association_rules(freq_items, metric="confidence", min_threshold=min_conf)

# เรียงตาม lift
rules = rules.sort_values(by="lift", ascending=False)

print("\n========== ASSOCIATION RULES ==========")
print(rules[['antecedents','consequents','support','confidence','lift']].head(10))

print("\nTotal Rules Generated:", len(rules))

# ==============================================
# 6) Spurious Rules
# ==============================================
spurious_rules = rules[rules['lift'] <= 1]

print("\n========== SPURIOUS RULES ==========")
print(spurious_rules)

print("Number of spurious rules:", len(spurious_rules))

# ==============================================
# 7) Surprising Patterns
# ==============================================
print("\n========== SURPRISING RULES ==========")
print(rules[['antecedents','consequents','support','confidence','lift']])

# ==============================================
# 8) Strong Rules
# ==============================================
strong_rules = rules[rules['lift'] > 1]

print("\n========== STRONG RULES ==========")
print(strong_rules[['antecedents','consequents','support','confidence','lift']].head(10))

========== RAW DATA ==========
              0          1           2                 3             4   \
0         shrimp    almonds     avocado    vegetables mix  green grapes   
1        burgers  meatballs        eggs               NaN           NaN   
2        chutney        NaN         NaN               NaN           NaN   
3         turkey    avocado         NaN               NaN           NaN   
4  mineral water       milk  energy bar  whole wheat rice     green tea   

                 5     6               7             8             9   \
0  whole weat flour  yams  cottage cheese  energy drink  tomato juice   
1               NaN   NaN             NaN           NaN           NaN   
2               NaN   NaN             NaN           NaN           NaN   
3               NaN   NaN             NaN           NaN           NaN   
4               NaN   NaN             NaN           NaN           NaN   

               10         11     12     13             14      15  \
0  low fat